# Selection editor

Example of a very basic selection editing interface:

<video controls src="./assets/selection_editor.webm">

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: selection editor")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [3]:
utilities.use_recording_commands()
utilities.selections.update_selection("root", renderer="ball and stick")

## Interaction & commands

Mode that adds/removes particles in the selection whenever an interaction begins:

In [4]:
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode


class ToggleMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        if "is_restraint" in interaction.properties:
            return

        selection = get_current_selection()
        particles = set(selection.get("selected", {}).get("particle_ids", []))

        if particles.intersection(interaction.particles):
            particles.difference_update(interaction.particles)
        else:
            particles.update(interaction.particles)

        utilities.selections.update_selection(
            selection["id"].removeprefix("selection."),
            particle_ids=list(map(int, particles)),
            renderer=selection.get("properties", {}).get("nanover.rendering.renderer", "liquorice"),
        )

        refresh_panels()


utilities.modes.add_normal_mode()
utilities.modes.add_mode(ToggleMode(), "toggle", icon="🫧")

In [5]:
from nanover.jupyter.utilities import make_id_generator

make_selection_id = make_id_generator("editor")

RENDERERS = ["liquorice", "ball and stick", "cartoon"]
CURRENT_SELECTION_INDEX = 0


def get_current_selection():
    selections = list(utilities.selections.all_prefixed())
    selections.sort()
    d = {key: value for key, value in utilities.selections.all_prefixed_items()}
    return d[selections[CURRENT_SELECTION_INDEX]]


def cycle_renderer():
    selection = get_current_selection()

    try:
        current = RENDERERS.index(selection.get("properties", {}).get("nanover.rendering.renderer", None))
        next = (current + 1) % len(RENDERERS)
    except ValueError:
        next = 0

    utilities.selections.update_selection(
        selection["id"].removeprefix("selection."),
        particle_ids=selection["selected"]["particle_ids"],
        renderer=RENDERERS[next],
    )
    refresh_panels()


def clear_particles():
    selection = get_current_selection()
    utilities.selections.update_selection(
        selection["id"].removeprefix("selection."),
        particle_ids=[],
        renderer=selection.get("properties", {}).get("nanover.rendering.renderer", None),
    )
    refresh_panels()


def prev_selection():
    global CURRENT_SELECTION_INDEX

    selections = utilities.selections.all_prefixed()
    CURRENT_SELECTION_INDEX = (CURRENT_SELECTION_INDEX - 1) % len(selections)

    refresh_panels()


def next_selection():
    global CURRENT_SELECTION_INDEX

    selections = utilities.selections.all_prefixed()
    CURRENT_SELECTION_INDEX = (CURRENT_SELECTION_INDEX + 1) % len(selections)

    refresh_panels()


def create_selection():
    global CURRENT_SELECTION_INDEX

    id = make_selection_id()

    selections = [*utilities.selections.all_prefixed(), id]
    selections.sort()
    CURRENT_SELECTION_INDEX = selections.index(id)

    utilities.selections.update_selection(id)
    utilities.notify_all(f"Deleted selection {id}")

    refresh_panels()


def delete_selection():
    global CURRENT_SELECTION_INDEX

    selections = list(utilities.selections.all_prefixed())
    selections.sort()
    id = selections.pop(CURRENT_SELECTION_INDEX)

    if id == "root":
        utilities.notify_all("Can't delete root selection.")
        return

    utilities.notify_all(f"Deleted selection {id}")

    utilities.selections.remove_selection(id)
    CURRENT_SELECTION_INDEX = max(CURRENT_SELECTION_INDEX - 1, 0)

    refresh_panels()


utilities.define_command("selections/cycle-renderer", handler=cycle_renderer)
utilities.define_command("selections/clear-particles", handler=clear_particles)
utilities.define_command("selections/prev", handler=prev_selection)
utilities.define_command("selections/next", handler=next_selection)
utilities.define_command("selections/create", handler=create_selection)
utilities.define_command("selections/delete", handler=delete_selection)

In [6]:
def refresh_panels():
    current = get_current_selection()

    utilities.panels.update_panel(
        "test",
        utilities.panels.header(label=f"Modify {current["id"]}"),
        utilities.panels.button(
            label=f"Renderer: {current.get("properties", {}).get("nanover.rendering.renderer", None)}",
            command="selections/cycle-renderer"),
        utilities.panels.button(
            label=f"Clear {len(current["selected"]["particle_ids"])} particles",
            command="selections/clear-particles",
        ),
        utilities.panels.header(label=f"Navigate"),
        utilities.panels.button(label="next selection", command="selections/next"),
        utilities.panels.button(label="prev selection", command="selections/prev"),
        utilities.panels.header(label=f"Add/remove"),
        utilities.panels.button(label="create selection", command="selections/create"),
        utilities.panels.button(label="delete selection", command="selections/delete"),
        label="Edit selections",
    )


refresh_panels()